In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ===============================
# Natural Language Processing with Disaster Tweets
# ===============================

# 1️⃣ Import libraries
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 2️⃣ Load datasets
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
sample_sub = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

# 3️⃣ Basic info
print(train_df.shape)
print(train_df.head())

# 4️⃣ Clean text function
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # remove URLs
    text = re.sub(r'\@w+|\#','', text)  # remove mentions and hashtags
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.strip()
    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

# 5️⃣ Split data for training and validation
X_train, X_val, y_train, y_val = train_test_split(
    train_df["clean_text"], train_df["target"], test_size=0.2, random_state=42
)

# 6️⃣ Convert text to numerical features using TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

# 7️⃣ Train a Logistic Regression model
model = LogisticRegression(max_iter=200)
model.fit(X_train_tfidf, y_train)

# 8️⃣ Validate model
val_pred = model.predict(X_val_tfidf)
f1 = f1_score(y_val, val_pred)
print("Validation F1 Score:", f1)

# 9️⃣ Predict on test data
test_pred = model.predict(X_test_tfidf)

# 🔟 Create submission file
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_pred
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv file created successfully!")
